## WP005 — Prior Loosening (structural resolution)

WP004 proved the model's gap to the market is a **resolution** problem — its match rankings are genuinely worse, not just mis-scaled — and no post-hoc recalibration touches it. WP003's over-shrinkage diagnosis points at the priors: team strengths are pulled toward the league mean and can't separate.

This work product loosens the shrinkage priors, one knob at a time, and asks whether that buys resolution back. Per-team `sigma` / partial pooling is deliberately **out of scope** — it's a bigger model change and gets its own work product (WP006).

**Three phases:**
1. **Prior-predictive triage** (no CV, minutes) — does the current prior even imply a realistic spread of team quality? Which loosened settings stay plausible?
2. **Screening CV** (reduced windows, one knob at a time) — which knob actually moves resolution / the gap to Pinnacle, without inflating the WP003 disagreement pattern (the overfitting tell)?
3. **Confirmation** — full 35-window CV on the 1–2 finalists + a cold held-out-season check the tuning never saw.

WP003's 401-match comparison stays the yardstick.

In [2]:
import json
import pickle
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pymc as pm

from football_model.data.prepare_model_data import prepare_model_data
from football_model.model.model import build_model
from football_model.model.predict import dc_outcome_probs
from football_model.types.model_data import ModelConfig

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP005 = REPO / 'work_products' / 'wp005_prior_loosening'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'

with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
print(len(windows), 'windows;', df_cv['team'].nunique(), 'teams total')

35 windows; 28 teams total


### Candidate configs

Each is the WP001 baseline (`clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True`) with **one** shrinkage knob loosened. Values chosen from the WP003 diagnosis, not a grid:

| arm | knob | baseline → candidate | effect |
|---|---|---|---|
| `baseline` | — | — | reference |
| `loose_init` | `init_scale` | 0.20 → 0.40 | wider t=0 team-strength prior (main cross-team spread term) |
| `loose_home` | `home_adv_sd` | 0.02 → 0.06 | lets per-team home edge vary from the league mean |
| `loose_sigma` | `sigma_att`/`sigma_def` | 0.008 → 0.020 | bigger AR(1) innovations — strengths move faster round-to-round |
| `loose_rho` | `rho_att_alpha`/`rho_def_alpha` | 29 → 12 | Beta(12,1) mean ≈0.92 vs ≈0.97 — less persistent, more mean-reverting |
| `loose_combo` | init+home+sigma | all three above | for Phase 3 if single knobs each help a little |

In [3]:
# Values are starting points — edit after reading the Phase 1 triage table
# (a quick smoke test already suggested init_scale=0.40 overshoots the real
# team-scoring spread ~2.5x to ~5x, so loose_init starts at 0.30 here).
ARMS = {
    'baseline':    {},
    'loose_init':  {'init_scale': 0.30},
    'loose_home':  {'home_adv_sd': 0.06},
    'loose_sigma': {'sigma_att': 0.020, 'sigma_def': 0.020},
    'loose_rho':   {'rho_att_alpha': 12.0, 'rho_def_alpha': 12.0},
    'loose_combo': {'init_scale': 0.30, 'home_adv_sd': 0.06, 'sigma_att': 0.020, 'sigma_def': 0.020},
}
BASE = dict(clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True)
for name, ov in ARMS.items():
    ModelConfig(**BASE, **ov)  # smoke: every override is a real field
print('all', len(ARMS), 'arm configs valid')

all 6 arm configs valid


## Phase 1 — Prior-predictive triage

Build the model on the last window's training data (full 28-team roster) and draw from the prior predictive. For each arm, summarise what the prior *implies* about the world, and check it against real EPL reference values. An arm whose prior implies absurd team-quality spread or scorelines is disqualified before any CV.

In [4]:
train_data = prepare_model_data(df_cv, max_round=windows[-1]['train_end'])
N_PP = 120   # prior-predictive draws — the AR(1) scan makes this the slow cell (~1-2 min/arm)

def prior_predictive_summary(overrides, draws=N_PP, seed=0):
    cfg = ModelConfig(**BASE, **overrides)
    with build_model(train_data, cfg):
        idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
    pr = idata.prior
    attack = pr['attack'].values[0]      # (draws, n_time, n_teams)
    defence = pr['defence'].values[0]
    home_adv = pr['home_adv'].values[0]  # (draws, n_teams)
    lam_h = pr['lambda_home'].values[0]  # (draws, n_obs)
    lam_a = pr['lambda_away'].values[0]

    t = attack.shape[1] - 1
    att_spread = attack[:, t, :].std(axis=1)                 # cross-team SD of attack, per draw
    def_spread = defence[:, t, :].std(axis=1)
    # implied home goals for a +2sd attack team vs a -2sd defence, per draw
    top_vs_bot = np.exp((attack[:, t, :].max(axis=1) - defence[:, t, :].min(axis=1)))
    league_lambda = np.exp(attack[:, t, :].mean(axis=1) - defence[:, t, :].mean(axis=1)
                           + home_adv.mean(axis=1))
    home_sd = home_adv.std(axis=1)
    # implied outcome rates from the prior-predictive lambdas
    from scipy.stats import poisson
    g = np.arange(11)
    def hda(lh, la):
        G = np.outer(poisson.pmf(g, lh), poisson.pmf(g, la))
        return np.tril(G, -1).sum(), np.trace(G), np.triu(G, 1).sum()
    rng = np.random.default_rng(seed)
    idx = rng.choice(lam_h.shape[1], size=min(60, lam_h.shape[1]), replace=False)
    hh = np.array([hda(lam_h[d, i], lam_a[d, i])
                   for d in range(min(60, draws)) for i in idx])
    hh = hh / hh.sum(axis=1, keepdims=True)
    return {
        'attack SD (cross-team)': att_spread,
        'defence SD (cross-team)': def_spread,
        'home_adv SD (cross-team)': home_sd,
        'top/bottom home-goals ratio': top_vs_bot,
        'league lambda (home)': league_lambda,
        'P(home win) implied': hh[:, 0],
        'P(draw) implied': hh[:, 1],
    }

rows = []
for name, ov in ARMS.items():
    s = prior_predictive_summary(ov)
    row = {'arm': name}
    for k, v in s.items():
        row[k] = f'{np.median(v):.3f} [{np.percentile(v, 5):.3f}, {np.percentile(v, 95):.3f}]'
    rows.append(row)
    print(name, 'done')
pp = pd.DataFrame(rows).set_index('arm')
pd.set_option('display.max_colwidth', None); pd.set_option('display.width', 200)
pp.T

/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_35899/1263264806.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]


baseline done


/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_35899/1263264806.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]


loose_init done


/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_35899/1263264806.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]


...

/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_35899/1263264806.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]


loose_rho done


/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_35899/1263264806.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]


loose_combo done


arm,baseline,loose_init,loose_home,loose_sigma,loose_rho,loose_combo
attack SD (cross-team),"0.213 [0.163, 0.289]","0.307 [0.242, 0.371]","0.213 [0.163, 0.289]","0.229 [0.173, 0.390]","0.204 [0.160, 0.264]","0.323 [0.251, 0.460]"
defence SD (cross-team),"0.209 [0.167, 0.278]","0.307 [0.244, 0.396]","0.209 [0.167, 0.278]","0.222 [0.172, 0.381]","0.201 [0.162, 0.265]","0.315 [0.248, 0.450]"
home_adv SD (cross-team),"0.011 [0.001, 0.036]","0.011 [0.001, 0.036]","0.034 [0.003, 0.108]","0.011 [0.001, 0.036]","0.011 [0.001, 0.036]","0.034 [0.003, 0.108]"
top/bottom home-goals ratio,"2.352 [1.899, 3.410]","3.404 [2.582, 5.498]","2.352 [1.899, 3.410]","2.533 [1.947, 5.121]","2.261 [1.878, 3.069]","3.685 [2.598, 6.656]"
league lambda (home),"1.142 [1.025, 1.260]","1.139 [0.999, 1.306]","1.143 [1.023, 1.260]","1.143 [1.007, 1.304]","1.143 [1.038, 1.259]","1.139 [0.980, 1.340]"
P(home win) implied,"0.378 [0.187, 0.633]","0.370 [0.130, 0.729]","0.380 [0.186, 0.638]","0.377 [0.169, 0.667]","0.378 [0.196, 0.620]","0.370 [0.119, 0.751]"
P(draw) implied,"0.276 [0.197, 0.356]","0.265 [0.148, 0.383]","0.276 [0.196, 0.357]","0.274 [0.180, 0.365]","0.277 [0.203, 0.354]","0.263 [0.137, 0.391]"


### Phase 1 reference values (real EPL, 2020-21 → 2025-26)

| Quantity | Real value | Read |
|---|---|---|
| P(home win) / draw / away | ~0.44 / 0.24 / 0.32 | prior median should bracket this, not sit far inside |
| League goals/game (one side) | ~1.45 | `league lambda (home)` should be near this |
| Top vs bottom team scoring ratio | ~2.5–3× (e.g. Man City ~2.6 g/g vs a relegated side ~0.9) | `top/bottom home-goals ratio` — if the baseline prior implies only ~1.5×, it's too tight |
| Per-team home-advantage spread | real teams' home edges differ by ~0.05–0.10 in log-λ | `home_adv SD` — baseline `home_adv_sd=0.02` almost certainly implies far less |

Fill the actual EPL numbers from `df_cv` in the next cell for a like-for-like check.

In [5]:
# actual EPL rates + team scoring spread from the CV data itself
home_rows = df_cv[df_cv['is_home'] == 1]
res = np.where(home_rows['goals_home'] > home_rows['goals_away'], 'H',
        np.where(home_rows['goals_home'] == home_rows['goals_away'], 'D', 'A'))
print('actual  P(H)/P(D)/P(A) =', np.round([(res == x).mean() for x in 'HDA'], 3))
print('actual  league goals/game (home side) =', round(home_rows['goals_home'].mean(), 3))
print('actual  league goals/game (away side) =', round(home_rows['goals_away'].mean(), 3))

# team scoring rate over the whole window, home+away
gf = pd.concat([
    df_cv[df_cv['is_home'] == 1].groupby('team')['goals_home'].mean(),
    df_cv[df_cv['is_home'] == 0].groupby('team')['goals_away'].mean(),
], axis=1).mean(axis=1).sort_values()
print(f'actual  team goals/game: min {gf.min():.2f} ({gf.index[0]}), max {gf.max():.2f} ({gf.index[-1]}), '
      f'ratio {gf.max() / gf.min():.2f}x')

actual  P(H)/P(D)/P(A) = [0.431 0.236 0.333]
actual  league goals/game (home side) = 1.557
actual  league goals/game (away side) = 1.332
actual  team goals/game: min 1.26 (WAT), max 2.00 (LUT), ratio 1.58x


## Phase 2 — Screening CV (reduced windows, one knob at a time)

Every 2nd window (18 of 35) — enough for a directional read at ~half the compute. Each arm writes `cv_checkpoint_<arm>.pkl` here; resumable. `baseline` reuses WP001's full checkpoint (filtered to the screening windows) so it isn't re-run.

**Heavy compute — run this yourself.** ~5 arms × 18 windows ≈ 90 fits.

In [9]:
SCREEN_WINDOWS = list(range(1, len(windows) + 1, 2))   # 1,3,5,...,35
DATA_PATH = WP001 / 'cv_shared_data.pkl'                # identical data/windows as WP001
WINDOW_TIMEOUT = 1200

def load_ckpt(p):
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

# seed baseline arm from WP001's finished checkpoint (filtered to screening windows)
base_ckpt = WP005 / 'cv_checkpoint_baseline.pkl'
if not base_ckpt.exists():
    w1 = load_ckpt(WP001 / 'cv_checkpoint.pkl')
    filt = {'results': [r for r in w1['results'] if r['window'] in SCREEN_WINDOWS],
            'cv_match_predictions': [m for m in w1['cv_match_predictions'] if m['window'] in SCREEN_WINDOWS]}
    pickle.dump(filt, open(base_ckpt, 'wb'))
    print('seeded baseline from WP001:', len(filt['results']), 'windows')

for arm, ov in ARMS.items():
    if arm == 'baseline':
        continue
    ckpt = WP005 / f'cv_checkpoint_{arm}.pkl'
    done = {r['window'] for r in load_ckpt(ckpt)['results']}
    todo = [w for w in SCREEN_WINDOWS if w not in done]
    print(f'\n### arm {arm}  overrides={ov}  ({len(done)}/{len(SCREEN_WINDOWS)} done, {len(todo)} to run)')
    for w in todo:
        print(f'  [{arm}] window {w}')
        try:
            subprocess.run(
                [sys.executable, str(SCRIPT),
                 '--data-path', str(DATA_PATH), '--checkpoint-path', str(ckpt),
                 '--window-index', str(w), '--config-json', json.dumps(ov)],
                timeout=WINDOW_TIMEOUT, check=True,
            )
        except subprocess.TimeoutExpired:
            print(f'  [{arm}] window {w} TIMEOUT — skipped, re-run cell to retry')
        except subprocess.CalledProcessError:
            print(f'  [{arm}] window {w} FAILED — skipped, re-run cell to retry')

print('\nscreening run complete')
for arm in ARMS:
    n = len(load_ckpt(WP005 / f'cv_checkpoint_{arm}.pkl')['results'])
    print(f'  {arm}: {n}/{len(SCREEN_WINDOWS)}')


### arm loose_init  overrides={'init_scale': 0.3}  (0/18 done, 18 to run)
  [loose_init] window 1
[window 1/35] training rounds 1-36 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:06<01:38, 38.60it/s]

Running chain 0:  10%|█         | 400/4000 [00:07<00:48, 74.80it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:08<00:32, 104.88it/s][A

Running chain 0:  20%|██        | 800/4000 [00:09<00:23, 134.09it/s][A

Running chain 1:  20%|██        | 800/4000 [00:10<00:26, 120.27it/s]

Running chain 1:  25%|██▌       | 1000/4000 [00:11<00:21, 141.17it/s]

Running chain 1:  30%|███       | 1200/4000 [00:12<00:16, 166.02it/s]

Running chain 1:  35%|███▌      | 1400/4000 [00:13<00:14, 181.98it/s]

Running chain 1:  40%|████      | 1600/4000 [00:14<00:12, 197.93it/s]

Running chain 1:  45%|████▌     | 1800/4000 [00:14<00:10, 208.49it/s]

Running chain 1:  50%|█████     | 2000/4000 [00:15<00:09, 216.33it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [00:16<00:07, 226.88i

[window 1] MAE=1.158 LL_improvement=1.08
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_loose_init.pkl
  [loose_init] window 3
[window 3/35] training rounds 1-46 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:07<01:50, 34.42it/s]

Running chain 0:  10%|█         | 400/4000 [00:08<00:56, 63.90it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:09<00:38, 87.59it/s]

Running chain 0:  20%|██        | 800/4000 [00:11<00:29, 110.07it/s][A

Running chain 0:  25%|██▌       | 1000/4000 [00:12<00:24, 123.11it/s]A

Running chain 1:  20%|██        | 800/4000 [00:12<00:33, 96.67it/s]

Running chain 1:  25%|██▌       | 1000/4000 [00:13<00:26, 111.59it/s]

Running chain 1:  30%|███       | 1200/4000 [00:14<00:21, 128.00it/s]

Running chain 1:  35%|███▌      | 1400/4000 [00:16<00:19, 135.79it/s]

Running chain 1:  40%|████      | 1600/4000 [00:17<00:15, 150.81it/s]

Running chain 1:  45%|████▌     | 1800/4000 [00:18<00:14, 152.08it/s]

Running chain 1:  50%|█████     | 2000/4000 [00:19<00:12, 157.68it/s

[window 3] MAE=0.662 LL_improvement=4.42
[window 3] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_loose_init.pkl
  [loose_init] window 5
[window 5/35] training rounds 1-56 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3})


...

[window 31] MAE=1.073 LL_improvement=1.22
[window 31] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_loose_combo.pkl
  [loose_combo] window 33
[window 33/35] training rounds 1-196 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:40<12:18,  5.14it/s]

Running chain 1:  10%|█         | 400/4000 [00:48<06:17,  9.55it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:57<04:18, 13.16it/s]

Running chain 1:  20%|██        | 800/4000 [01:05<03:20, 15.95it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:15<02:50, 17.55it/s]

Running chain 1:  30%|███       | 1200/4000 [01:23<02:26, 19.06it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:32<02:07, 20.42it/s]

Running chain 1:  40%|████      | 1600/4000 [01:40<01:52, 21.37it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:49<01:39, 22.15it/s]

Running chain 1:  50%|█████     | 2000/4000 [01:57<01:29, 22.44it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:06<01:19, 22.63it/s]

Running chain 1:  

[window 33] MAE=0.840 LL_improvement=-0.08
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_loose_combo.pkl
  [loose_combo] window 35
[window 35/35] training rounds 1-206 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:46<14:01,  4.51it/s]

Running chain 1:  10%|█         | 400/4000 [00:55<07:12,  8.33it/s]

Running chain 0:  15%|█▌        | 600/4000 [01:06<05:04, 11.18it/s]

Running chain 0:  20%|██        | 800/4000 [01:16<03:59, 13.39it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:27<03:22, 14.82it/s]

Running chain 0:  30%|███       | 1200/4000 [01:37<02:51, 16.32it/s]

Running chain 0:  35%|███▌      | 1400/4000 [01:47<02:29, 17.33it/s]

Running chain 0:  40%|████      | 1600/4000 [01:57<02:12, 18.18it/s]

Running chain 0:  45%|████▌     | 1800/4000 [02:07<01:57, 18.70it/s]

Running chain 0:  50%|█████     | 2000/4000 [02:17<01:45, 19.02it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [02:27<01:33, 19.25it/s]

Running chain 0:  60%|██████    | 2400/4000 [02:37<01:22, 19.46it/s]

Running

[window 35] MAE=0.901 LL_improvement=0.84
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_loose_combo.pkl

screening run complete
  baseline: 18/18
  loose_init: 18/18
  loose_home: 18/18
  loose_sigma: 18/18
  loose_rho: 18/18
  loose_combo: 18/18


### Phase 2 analysis — resolution + gap to Pinnacle per arm

Reuses WP003's odds join (`../wp003_bookmaker_benchmark/odds_raw.pkl`) and scoring. For each arm: pooled RPS on the screening matches, RPS gap to Pinnacle, and the WP003 disagreement-decile pattern (a looser prior that makes the gap grow *faster* with disagreement is overfitting, not resolution — disqualify).

In [10]:
odds_raw = pd.read_pickle(WP003 / 'odds_raw.pkl')
first_round = df_cv.groupby('season')['round'].min().to_dict()
CODE_TO_FD = {'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace', 'EVE': 'Everton',
    'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds', 'LEI': 'Leicester', 'LIV': 'Liverpool',
    'LUT': 'Luton', 'MCI': 'Man City', 'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich',
    'NOT': "Nott'm Forest", 'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland',
    'TOT': 'Tottenham', 'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves'}
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)

def fixtures_for(ckpt, windows_list=None):
    """windows_list: the windows a checkpoint's '--window-index' values refer
    to. Defaults to the global 35-window WP001 list (correct for Phase 2/3's
    full-CV checkpoints, which all share that same data file) — but the
    Phase 3 held-out checkpoint is built from its OWN one-window file
    (`ho_data`, holding just `[holdout_window]`), so its window "1" means
    `holdout_window`, not `windows[0]`. Passing the wrong list here silently
    reconstructs the wrong fixtures; the assert below is what catches it."""
    windows_list = windows if windows_list is None else windows_list
    mp = ckpt['cv_match_predictions']
    rows = []
    for w in sorted({m['window'] for m in mp}):
        win = windows_list[w - 1]
        sel = df_sorted[(df_sorted['is_home'] == 1) & (df_sorted['round'] >= win['test_start'])
                        & (df_sorted['round'] <= win['test_end'])]
        for (_, r), m in zip(sel.iterrows(), [x for x in mp if x['window'] == w]):
            assert int(r['goals_home']) == m['goals_home'] and int(r['goals_away']) == m['goals_away']
            rows.append({'date': pd.Timestamp(r['datetime']).normalize(),
                         'home_fd': CODE_TO_FD[r['team']], 'away_fd': CODE_TO_FD[r['opp_team']],
                         'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'],
                         'rho_dc': m.get('rho_dc'), 'goals_home': m['goals_home'], 'goals_away': m['goals_away']})
    d = pd.DataFrame(rows)
    P = np.array([dc_outcome_probs(x.lambda_home, x.lambda_away, rho=x.rho_dc) for x in d.itertuples()])
    d[['p_home', 'p_draw', 'p_away']] = P
    d['result'] = np.where(d['goals_home'] > d['goals_away'], 'H',
                    np.where(d['goals_home'] == d['goals_away'], 'D', 'A'))
    return d.merge(odds_raw, left_on=['date', 'home_fd', 'away_fd'],
                   right_on=['Date', 'HomeTeam', 'AwayTeam'], how='inner')

def rps_row(ph, pdw, pa, a):
    c1, c2 = ph, ph + pdw
    e1 = 1.0 if a == 'H' else 0.0
    e2 = 1.0 if a in ('H', 'D') else 0.0
    return 0.5 * ((c1 - e1) ** 2 + (c2 - e2) ** 2)

def boot(v, n=4000, seed=0):
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    b = np.array([rng.choice(v, len(v), replace=True).mean() for _ in range(n)])
    return v.mean(), *np.percentile(b, [2.5, 97.5])

def devig(o):
    inv = 1 / np.asarray(o, float); return inv / inv.sum()

rows = []
for arm in ARMS:
    ck = WP005 / f'cv_checkpoint_{arm}.pkl'
    if not ck.exists() or len(load_ckpt(ck)['results']) == 0:
        print(f'{arm}: not run yet'); continue
    d = fixtures_for(load_ckpt(ck))
    pin_ok = d[['PSCH', 'PSCD', 'PSCA']].notna().all(axis=1) if 'PSCH' in d else pd.Series(False, index=d.index)
    d = d[pin_ok].copy()
    Pin = np.array([devig(r) for r in d[['PSCH', 'PSCD', 'PSCA']].to_numpy()])
    d['rps_m'] = [rps_row(x.p_home, x.p_draw, x.p_away, x.result) for x in d.itertuples()]
    d['rps_p'] = [rps_row(Pin[i, 0], Pin[i, 1], Pin[i, 2], d['result'].iloc[i]) for i in range(len(d))]
    d['disag'] = np.abs(d['p_home'].values - Pin[:, 0])
    g_all, lo, hi = boot((d['rps_m'] - d['rps_p']).values)
    top = d[d['disag'] >= d['disag'].quantile(0.75)]
    g_top, _, _ = boot((top['rps_m'] - top['rps_p']).values)
    rows.append({'arm': arm, 'n': len(d), 'model RPS': round(d['rps_m'].mean(), 4),
                 'gap vs Pinnacle': f'{g_all:+.4f}', 'CI': f'[{lo:+.4f},{hi:+.4f}]',
                 'gap top-25% disagree': f'{g_top:+.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

        arm   n  model RPS gap vs Pinnacle                CI gap top-25% disagree
   baseline 195     0.1916         +0.0129 [+0.0058,+0.0200]              +0.0455
 loose_init 195     0.1909         +0.0123 [+0.0051,+0.0194]              +0.0438
 loose_home 195     0.1914         +0.0128 [+0.0056,+0.0199]              +0.0396
loose_sigma 195     0.1912         +0.0126 [+0.0056,+0.0195]              +0.0393
  loose_rho 195     0.1919         +0.0132 [+0.0059,+0.0202]              +0.0466
loose_combo 195     0.1905         +0.0118 [+0.0049,+0.0188]              +0.0400


## Phase 3 — Confirmation (finalists only)

After Phase 2, put the winning arm name(s) in `FINALISTS`, run the **full** 35-window CV for each, then the cold held-out-season check: train through 2024-25, score 2025-26 — a split the screening never touched.

**Heavy compute — run yourself.**

In [12]:
FINALISTS = ["loose_combo"]   # <- set after Phase 2, e.g. ['loose_init', 'loose_combo']

FULL_WINDOWS = list(range(1, len(windows) + 1))
for arm in FINALISTS:
    ov = ARMS[arm]
    ckpt = WP005 / f'cv_checkpoint_full_{arm}.pkl'
    done = {r['window'] for r in load_ckpt(ckpt)['results']}
    for w in [x for x in FULL_WINDOWS if x not in done]:
        print(f'[full {arm}] window {w}')
        try:
            subprocess.run([sys.executable, str(SCRIPT), '--data-path', str(DATA_PATH),
                            '--checkpoint-path', str(ckpt), '--window-index', str(w),
                            '--config-json', json.dumps(ov)], timeout=WINDOW_TIMEOUT, check=True)
        except subprocess.SubprocessError as e:
            print(f'  window {w} problem ({type(e).__name__}) — re-run to retry')
print('finalist full-CV run complete' if FINALISTS else 'set FINALISTS first')

[full loose_combo] window 1
[window 1/35] training rounds 1-36 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:07<01:42, 37.25it/s]

Running chain 0:  10%|█         | 400/4000 [00:08<00:53, 66.76it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:09<00:37, 91.47it/s]

Running chain 1:  25%|██▌       | 1000/4000 [00:11<00:20, 143.52it/s]

Running chain 0:  25%|██▌       | 1000/4000 [00:11<00:23, 130.03it/s]

Running chain 0:  30%|███       | 1200/4000 [00:12<00:19, 143.39it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:14<00:16, 154.51it/s]

Running chain 0:  40%|████      | 1600/4000 [00:14<00:14, 169.05it/s]

Running chain 0:  45%|████▌     | 1800/4000 [00:16<00:12, 176.43it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [00:18<00:11, 159.51it/s]

Running chain 1:  60%|██████    | 2400/4000 [00:19<00:10, 155.08it/s]

Running c

[window 1] MAE=1.159 LL_improvement=1.05
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_full_loose_combo.pkl
[full loose_combo] window 2
[window 2/35] training rounds 1-41 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:  10%|█         | 400/4000 [00:08<00:52, 68.20it/s]

Running chain 1:  10%|█         | 400/4000 [00:08<00:55, 65.39it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:09<00:37, 91.40it/s]

Running chain 1:  20%|██        | 800/4000 [00:10<00:28, 111.17it/s]

Running chain 1:  30%|███       | 1200/4000 [00:13<00:20, 137.23it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:14<00:18, 139.08it/s]

Running chain 0:  40%|████      | 1600/4000 [00:16<00:16, 146.37it/s]

Running chain 0:  45%|████▌     | 1800/4000 [00:17<00:14, 153.40it/s]

Running chain 0:  50%|█████     | 2000/4000 [00:18<00:13, 152.69it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [00:19<00:10, 168.56it/s]

Running chain 1:  65%|██████▌   | 2600/4000 [00:21<00:07, 194.19it/s]

Running chain 1:  75%|███████▌  | 3000/4000 [00:23<00:04, 208.06it/s]

[window 2] MAE=1.287 LL_improvement=1.48
[window 2] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_full_loose_combo.pkl
[full loose_combo] window 3
[window 3/35] training rounds 1-46 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


...

[window 33] MAE=0.840 LL_improvement=-0.08
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_full_loose_combo.pkl
[full loose_combo] window 34
[window 34/35] training rounds 1-201 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:41<12:29,  5.07it/s]

Running chain 1:  10%|█         | 400/4000 [00:49<06:23,  9.38it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:58<04:21, 13.00it/s]

Running chain 1:  20%|██        | 800/4000 [01:07<03:25, 15.55it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:16<02:56, 17.04it/s]

Running chain 1:  30%|███       | 1200/4000 [01:26<02:32, 18.34it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:34<02:12, 19.55it/s]

Running chain 1:  40%|████      | 1600/4000 [01:43<01:58, 20.33it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:52<01:44, 20.96it/s]

Running chain 1:  50%|█████     | 2000/4000 [02:02<01:36, 20.66it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:11<01:25, 21.10it/s]

Running chain 1:  

[window 34] MAE=0.865 LL_improvement=0.86
[window 34] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_full_loose_combo.pkl
[full loose_combo] window 35
[window 35/35] training rounds 1-206 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:48<14:49,  4.27it/s]

Running chain 0:  10%|█         | 400/4000 [00:58<07:36,  7.89it/s]

Running chain 0:  15%|█▌        | 600/4000 [01:09<05:15, 10.76it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:28<03:22, 14.85it/s]

Running chain 1:  30%|███       | 1200/4000 [01:39<02:52, 16.21it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:49<02:31, 17.18it/s]

Running chain 1:  40%|████      | 1600/4000 [01:59<02:14, 17.90it/s]

Running chain 1:  45%|████▌     | 1800/4000 [02:09<01:58, 18.56it/s]

Running chain 1:  50%|█████     | 2000/4000 [02:19<01:46, 18.71it/s]

Running chain 2:  45%|████▌     | 1800/4000 [02:20<01:59, 18.46it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:29<01:34, 19.11it/s]

Running chain 1: 

[window 35] MAE=0.901 LL_improvement=0.84
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_full_loose_combo.pkl
finalist full-CV run complete


### Phase 3 — held-out-season check

Single train/test split, no walk-forward: train on every round up to the end of 2024-25, predict every 2025-26 match. This is the honest generalisation check — the screening metric (gap to Pinnacle on the 401 walk-forward matches) is what was tuned, so a finalist has to also hold up here.

In [13]:
last_season = sorted(df_cv['season'].unique())[-1]
train_end_round = int(df_cv[df_cv['season'] != last_season]['round'].max())
test_rounds = sorted(df_cv[df_cv['season'] == last_season]['round'].unique())
holdout_window = {'train_start': 1, 'train_end': train_end_round,
                  'test_start': int(test_rounds[0]), 'test_end': int(test_rounds[-1])}
print('held-out window:', holdout_window, f'({len(test_rounds)} rounds of {last_season})')

# save a one-window shared file + drive run_cv_window for baseline vs each finalist
ho_data = WP005 / 'holdout_shared_data.pkl'
pickle.dump({'df_cv': df_cv, 'windows': [holdout_window]}, open(ho_data, 'wb'))
for arm in (['baseline'] + FINALISTS):
    ov = ARMS[arm]
    ckpt = WP005 / f'cv_checkpoint_holdout_{arm}.pkl'
    if load_ckpt(ckpt)['results']:
        print(f'{arm}: holdout already done'); continue
    print(f'[holdout {arm}]')
    subprocess.run([sys.executable, str(SCRIPT), '--data-path', str(ho_data),
                    '--checkpoint-path', str(ckpt), '--window-index', '1',
                    '--config-json', json.dumps(ov)], timeout=WINDOW_TIMEOUT, check=True)

# then score each with fixtures_for(...) + the odds join, same as Phase 2 analysis.
# NOTE: this checkpoint's window "1" means `holdout_window` (test rounds
# 173-208-ish), NOT windows[0] of the global 35-window list (test round 37)
# — must pass windows_list=[holdout_window] explicitly, or fixtures_for
# reconstructs the wrong fixtures entirely (caught by its internal assert).
for arm in (['baseline'] + FINALISTS):
    ck = WP005 / f'cv_checkpoint_holdout_{arm}.pkl'
    if not load_ckpt(ck)['results']:
        continue
    d = fixtures_for(load_ckpt(ck), windows_list=[holdout_window])
    pin_ok = d[['PSCH', 'PSCD', 'PSCA']].notna().all(axis=1)
    d = d[pin_ok].copy()
    Pin = np.array([devig(r) for r in d[['PSCH', 'PSCD', 'PSCA']].to_numpy()])
    rm = np.array([rps_row(x.p_home, x.p_draw, x.p_away, x.result) for x in d.itertuples()])
    rp = np.array([rps_row(Pin[i, 0], Pin[i, 1], Pin[i, 2], d['result'].iloc[i]) for i in range(len(d))])
    g, lo, hi = boot(rm - rp)
    print(f'{arm:>14}  n={len(d)}  model RPS {rm.mean():.4f}  gap vs Pinnacle {g:+.4f}  CI [{lo:+.4f},{hi:+.4f}]')

held-out window: {'train_start': 1, 'train_end': 172, 'test_start': 173, 'test_end': 208} (36 rounds of 2025)
baseline: holdout already done
[holdout loose_combo]
[window 1/1] training rounds 1-172 (use_xg=True, use_dc=True, overrides={'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:41<12:39,  5.00it/s]

Running chain 1:  10%|█         | 400/4000 [00:50<06:31,  9.19it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:59<04:26, 12.75it/s]

Running chain 1:  20%|██        | 800/4000 [01:07<03:25, 15.56it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:16<02:48, 17.78it/s]

Running chain 1:  30%|███       | 1200/4000 [01:24<02:25, 19.21it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:33<03:14, 15.46it/s]

Running chain 0:  30%|███       | 1200/4000 [01:42<02:40, 17.41it/s]

Running chain 0:  35%|███▌      | 1400/4000 [01:50<02:16, 19.12it/s]

Running chain 1:  50%|█████     | 2000/4000 [01:59<01:29, 22.22it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:07<01:20, 22.45it/s]

Running chain 0:  

[window 1] MAE=0.893 LL_improvement=26.56
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp005_prior_loosening/cv_checkpoint_holdout_loose_combo.pkl
      baseline  n=210  model RPS 0.2017  gap vs Pinnacle +0.0023  CI [-0.0048,+0.0098]
   loose_combo  n=210  model RPS 0.2013  gap vs Pinnacle +0.0020  CI [-0.0052,+0.0095]


## Results

See `README.md` for the full writeup. Summary: no single shrinkage-prior knob (`init_scale`, `home_adv_sd`, `sigma_att/def`, `rho_*_alpha`) shows a real effect on resolution, individually. The combined arm (`loose_combo`) came out marginally better than baseline on all three tests run here (screen, full 35-window CV, held-out 2025-26 season) — never statistically significant on its own, but consistently in the same favourable direction across every cut of the data. Not a basis to ship, but worth carrying `loose_combo`'s settings forward as the starting point for WP006 (per-team `sigma`) rather than reverting to the original tight priors.